# DuoRec + CORE — RetailRocket (Colab)

**Quan trọng:** Colab chỉ có **1 GPU**. Không chạy DuoRec và CORE **song song** — tranh VRAM/compute, log đứng ở batch đầu rất lâu, hoặc session disconnect sẽ kill process.

Notebook này chạy **tuần tự**: DuoRec xong → CORE.

Patch `torch.load` (PyTorch 2.6+) được áp dụng tự động qua `ncs_numpy_compat.py` — tránh lỗi test CORE sau training.

Ước tính (T4, full retailrocket):
- DuoRec 50 epoch: ~50–80 giờ (3734 batch/epoch)
- CORE trm ~42 epoch: ~15–20 phút

In [ ]:
# ===== CONFIG =====
REPO_DIR = "/content/test_all"   # clone/upload repo vào đây
DATASET = "retailrocket"

# Smoke test nhanh (1 epoch, vài phút): đặt True
SMOKE = False

# Preprocess + symlink data (chạy 1 lần nếu chưa có Data/)
RUN_PREPROCESS = False

In [ ]:
import os, sys, subprocess
from datetime import datetime
from pathlib import Path

repo = Path(REPO_DIR)
assert (repo / "nhom3/DuoRec/run_seq.py").exists(), f"Không thấy repo tại {repo}"
assert (repo / "nhom3/CORE/main.py").exists(), f"Không thấy repo tại {repo}"
os.chdir(repo)
sys.path.insert(0, str(repo))

from ncs_numpy_compat import apply_numpy_recbole_compat, apply_torch_load_compat
apply_numpy_recbole_compat()
apply_torch_load_compat()

get_ipython().system("pip install -q recbole scipy")

if RUN_PREPROCESS:
    get_ipython().system(f"{sys.executable} nhom3/DuoRec/preprocess_retailrocket.py")
    get_ipython().system(f"{sys.executable} nhom3/CORE/preprocess_retailrocket.py")

get_ipython().system("mkdir -p nhom3/DuoRec/dataset nhom3/CORE/dataset")
get_ipython().system(f"ln -sfn ../../../Data/DuoRec/{DATASET} nhom3/DuoRec/dataset/{DATASET}")
get_ipython().system(f"ln -sfn ../../../Data/CORE/{DATASET} nhom3/CORE/dataset/{DATASET}")

smoke_env = {"NCS_SMOKE": "1"} if SMOKE else {}


def run_one(name: str, script: str, *extra_args: str) -> int:
    ts = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
    log = repo / "Log" / name / DATASET / f"log-{ts}.log"
    log.parent.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, "-u", script, *extra_args]
    print("=" * 60)
    print(f"START {name}  {datetime.now().isoformat()}")
    print("CMD:", " ".join(cmd))
    print("LOG:", log)
    env = os.environ.copy()
    env.update(smoke_env)
    with open(log, "w", buffering=1) as f:
        proc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=env)
    print(f"DONE {name}  exit={proc.returncode}  {datetime.now().isoformat()}")
    print("--- tail log ---")
    get_ipython().system(f"tail -40 {log}")
    return proc.returncode

# Tuần tự — KHÔNG chạy song song trên 1 GPU Colab
# DuoRec log: mỗi ~50 batch. Trên T4, batch 0→50 có thể mất 30–60 phút — không phải treo.
ed = run_one(
    "DuoRec",
    "nhom3/DuoRec/run_seq.py",
    "--dataset", DATASET,
    "--model", "DuoRec",
    "--config_files", "seq.yaml DuoRec.yaml",
)
if ed != 0:
    raise SystemExit(f"DuoRec failed with exit={ed}")

ec = run_one(
    "CORE",
    "nhom3/CORE/main.py",
    "--model", "trm",
    "--dataset", DATASET,
)
if ec != 0:
    raise SystemExit(f"CORE failed with exit={ec}")

print("ALL DONE")